In [12]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [13]:
import numpy as np
#SEED = 45
#np.random.seed(SEED)

from models import GibbsSamplerLLFM
from data import generate_parametric
from post_process import frequency_align, hungarian_align
from sklearn.metrics import mean_squared_error

In [14]:
T = 500
S = 4
K = 8
alpha = 5.0
rho = 0.2
sigma_w = 1.0
mu_b = 0.0
sigma_b = 1.0
n_iter = 1500
burn = 500
n_subsample = 200
Y, Z_true, W_true, b_true, p_true = generate_parametric(T=T, S=S, K=K, alpha=alpha, rho=rho, sigma_w=sigma_w, mu_b=mu_b, sigma_b=sigma_b)
print("Latent probabilities:", p_true)
print("True weight matrix W_true:", W_true)
print("True bias vector b_true:", b_true)

Latent probabilities: [0.7566616  0.47177703 0.1131097  0.24310745 0.75446757 0.40747821
 0.74316018 0.45717312]
True weight matrix W_true: [[ 0.         -0.          0.         -0.        ]
 [ 0.          0.          0.         -0.        ]
 [-0.          0.61651998 -0.          0.        ]
 [ 0.         -0.28186736  0.30518568  0.        ]
 [ 0.         -0.28970182 -0.          0.        ]
 [-0.         -0.         -0.         -0.        ]
 [ 0.24831583 -0.          0.          0.        ]
 [ 0.         -0.          0.          0.        ]]
True bias vector b_true: [ 1.16074123 -0.44710076  0.88262425 -0.57891235]


In [15]:
np.linalg.norm(W_true, ord='fro')

np.float64(0.8356268855497809)

In [16]:
sampler = GibbsSamplerLLFM(
    Data=Y,
    K=K,
    alpha=alpha,
    rho=rho,
    sigma_w=sigma_w,
    mu_b=mu_b,
    sigma_b=sigma_b,
    n_iter=n_iter,
    burn=burn,
    n_subsample=n_subsample
)

sampler.run(verbose=True)
sampler.get_posterior_samples()

Iteration 25: ||W||_F = 4.0222
Iteration 50: ||W||_F = 4.2533
Iteration 75: ||W||_F = 4.3708
Iteration 100: ||W||_F = 3.3085
Iteration 125: ||W||_F = 2.0840
Iteration 150: ||W||_F = 2.1846
Iteration 175: ||W||_F = 2.8605
Iteration 200: ||W||_F = 4.1803
Iteration 225: ||W||_F = 2.8205
Iteration 250: ||W||_F = 2.8057
Iteration 275: ||W||_F = 2.4431
Iteration 300: ||W||_F = 2.0403
Iteration 325: ||W||_F = 0.5335
Iteration 350: ||W||_F = 3.0149
Iteration 375: ||W||_F = 0.7043
Iteration 400: ||W||_F = 1.1196
Iteration 425: ||W||_F = 1.4930
Iteration 450: ||W||_F = 2.3782
Iteration 475: ||W||_F = 2.1810
Iteration 500: ||W||_F = 1.4701
Iteration 525: ||W||_F = 2.7869
Iteration 550: ||W||_F = 2.8857
Iteration 575: ||W||_F = 2.8002
Iteration 600: ||W||_F = 2.8217
Iteration 625: ||W||_F = 3.5724
Iteration 650: ||W||_F = 3.4191
Iteration 675: ||W||_F = 3.8072
Iteration 700: ||W||_F = 4.5899
Iteration 725: ||W||_F = 3.6165
Iteration 750: ||W||_F = 4.4372
Iteration 775: ||W||_F = 3.4192
Iteration 8

In [17]:
Z_samples = sampler.good_samples_Z      # (n_samples, T, K)
W_samples = sampler.good_samples_W      # (n_samples, K, S)
b_samples = sampler.good_samples_b      # (n_samples, S)


Z_aligned_samples, W_aligned_samples, b_aligned_samples = hungarian_align(Z_samples, W_samples, b_samples, W_true)

W_mean = W_aligned_samples.mean(axis=0)
Z_mean = Z_aligned_samples.mean(axis=0)
p_mean = Z_mean.mean(axis=0)
b_mean = b_aligned_samples.mean(axis=0)

In [18]:
print("True bias vector b_true:", b_true)
print("Estimated bias vector b_mean:", b_mean)
print("True weight matrix W_true:", W_true)
print("Estimated weight matrix W_aligned:", W_mean)
print("Latent probabilities p_true:", p_true)
print("Estimated latent probabilities p_aligned:", p_mean)

True bias vector b_true: [ 1.16074123 -0.44710076  0.88262425 -0.57891235]
Estimated bias vector b_mean: [ 0.90766128 -0.91732493  0.7930807  -0.28036446]
True weight matrix W_true: [[ 0.         -0.          0.         -0.        ]
 [ 0.          0.          0.         -0.        ]
 [-0.          0.61651998 -0.          0.        ]
 [ 0.         -0.28186736  0.30518568  0.        ]
 [ 0.         -0.28970182 -0.          0.        ]
 [-0.         -0.         -0.         -0.        ]
 [ 0.24831583 -0.          0.          0.        ]
 [ 0.         -0.          0.          0.        ]]
Estimated weight matrix W_aligned: [[ 0.00160475  0.00386511 -0.00238786 -0.00139989]
 [-0.01749076  0.00323874 -0.00154409 -0.02708916]
 [ 0.2626501   0.66559247 -0.25514737 -0.05190966]
 [ 0.02157379 -0.23210053  0.78499885 -0.26293413]
 [ 0.09890279 -0.28818776 -0.18502571 -0.03544289]
 [ 0.01297215  0.0115205  -0.03126831 -0.02071043]
 [ 0.70306711  0.08550664 -0.30209859  0.05577001]
 [ 0.00818129  0.

In [19]:
print("W elementwise MSE:")
print((W_true - W_mean)**2)

print("Bias elementwise MSE:")
print((b_true - b_mean)**2)

print("p elementwise MSE:")
print((p_true - p_mean)**2)

W elementwise MSE:
[[2.57520664e-06 1.49390705e-05 5.70188485e-06 1.95969538e-06]
 [3.05926771e-04 1.04894534e-05 2.38422490e-06 7.33822396e-04]
 [6.89850746e-02 2.40810909e-03 6.51001789e-02 2.69461248e-03]
 [4.65428356e-04 2.47673702e-03 2.30220678e-01 6.91343569e-02]
 [9.78176196e-03 2.29238418e-06 3.42345138e-02 1.25619869e-03]
 [1.68276576e-04 1.32721823e-04 9.77707036e-04 4.28921781e-04]
 [2.06798729e-01 7.31138619e-03 9.12635598e-02 3.11029415e-03]
 [6.69334852e-05 1.08308792e-03 2.87164021e-04 1.14189732e-04]]
Bias elementwise MSE:
[0.06404946 0.22111077 0.00801805 0.08913084]
p elementwise MSE:
[0.28435727 0.09050894 0.1543785  0.05216316 0.11681874 0.04375134
 0.08044608 0.03500009]


In [20]:
print("W MSE:", mean_squared_error(W_true, W_mean))
print("p MSE:", mean_squared_error(p_true, p_mean))
print("Bias MSE:", mean_squared_error(b_true, b_mean))


W MSE: 0.024986897266267183
p MSE: 0.10717801471471577
Bias MSE: 0.09557727924912918


In [21]:

def normalized_mse(true, est):
    return np.sum((true - est)**2) / np.sum(true**2)

print("W NMSE:", normalized_mse(W_true, W_mean))
print("Bias NMSE:", normalized_mse(b_true, b_mean))
print("p NMSE:", normalized_mse(p_true, p_mean))

W NMSE: 1.1450844059671772
Bias NMSE: 0.14365047157958394
p NMSE: 0.3627681558790249


In [22]:
def corrcoef_flat(a, b):
    return np.corrcoef(a.flatten(), b.flatten())[0,1]

print("W correlation:", corrcoef_flat(W_true, W_mean))
print("Bias correlation:", np.corrcoef(b_true, b_mean)[0,1])
print("p correlation:", np.corrcoef(p_true, p_mean)[0,1])

W correlation: 0.8109810950902401
Bias correlation: 0.9330840268249352
p correlation: -0.2544774322224302
